#  Exercise 3: Streaming Aggregations

The purpose of this exercise is to apply an aggregation directly in a stream, using time windows

In [0]:
%sql
-- Step 0: Volumne Creation
CREATE VOLUME IF NOT EXISTS workspace.default.streaming_demo;
-- Since the volume is created, we can use it as a location for our tables

In [0]:
# Step 1: We have to create the folder for the checkpoint
dbutils.fs.mkdirs("/Volumes/workspace/default/streaming_demo/chk3")

True

In [0]:
# Step 2: Aggregation creation, in this case the time window is 1 minute (We will use the stream defined in exercise 1)
from pyspark.sql.functions import window, col
from pyspark.sql import functions as F

df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/streaming_demo/schema")
    .load("/Volumes/workspace/default/streaming_demo/input")
    .withColumn("timestamp", F.current_timestamp())
)

agg = (
    df_stream
    .groupBy(window(col("timestamp"), "1 minute"))
    .count()
)

In [0]:
%sql
-- Step 3: Streaming aggregations table creation
CREATE TABLE IF NOT EXISTS workspace.default.streaming_counts (
    window STRUCT<start: TIMESTAMP, end: TIMESTAMP>,
    count BIGINT
);

In [0]:
# Step 4: Stream writing
(
    agg.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/chk3")
        .table("workspace.default.streaming_counts")
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4564028209759911>, line 7
      1 # Step 4: Stream writing
      2 (
      3     agg.writeStream
      4         .format("delta")
      5         .outputMode("complete")
      6         .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/chk3")
----> 7         .table("workspace.default.streaming_counts")
      8 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:737, in DataStreamWriter.table(self, tableName)
    735 def table(self, tableName: str) -> "StreamingQuery":
    736     """Alias for the toTable API"""
--> 737     return self.toTable(tableName)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:748, in DataStreamWriter.toTable(self, tableName, format, outputMode, partitionBy, queryName, **options)


The error above is self explanatory due to the limitations of Databricks free edition, it doesn't allow triggers based in time, thus we will use the option `availableNow = True`

In [0]:
(
    agg.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/chk3")
        .trigger(availableNow=True)   # ✔ ADAPTACIÓN CLAVE
        .table("workspace.default.streaming_counts")
)

Finally we validate the stream was written in the table 

In [0]:
%sql
SELECT * FROM workspace.default.streaming_counts;

window,count
"List(2026-04-17T19:18:00.000Z, 2026-04-17T19:19:00.000Z)",6
